In [ ]:
import os
import json
import pandas as pd
import numpy as np
import ast

# --- Configuration ---
data_dir = '/media/hcv530/T7/garment_folding_data/real_world/'

method_label = {
    'PlaNet-ClothPick': 'planet_clothpick_single_picker_single_primitive_multi_longsleeve_flattening_longsleeve_all_garment_5k_eps',
    'LaGarNet': 'planet_clothpick_single_picker_single_primitive_multi_longsleeve_flattening_longsleeve_all_garment_5k_eps', # Update if LaGarNet has a different path
    'Human': 'planet_clothpick_single_picker_single_primitive_multi_longsleeve_flattening_longsleeve_all_garment_5k_eps'     # Update if Human has a different path
}

garment_label = {
    'Longsleeve': 'small-blue-top',
    'Trousers': 'small-blue-trouser',
    'Dress': 'small-blue-dress',
    'Skirt': 'small-yellow-skirt'
}

# Reverse mapping to look up category by garment_id from info.json
id_to_category = {v: k for k, v in garment_label.items()}

steps_to_evaluate = [10, 20] # The two halves of your table

# --- Data Structure to hold parsed metrics ---
results = {m: {s: {g: {'NC': [], 'NI': [], 'MaxIoU': [], 'success_90_80': 0, 'success_90_0': 0, 'total': 0} 
                   for g in list(garment_label.keys()) + ['Real Total']} 
               for s in steps_to_evaluate} 
           for m in method_label.keys()}

# ---> NEW: Dictionary to track the lists of episode IDs <---
trial_counts = {m: {g: [] for g in garment_label.keys()} for m in method_label.keys()}

# --- Step 1: Parse the Data ---
for method, folder_name in method_label.items():
    performance_dir = os.path.join(data_dir, folder_name, 'eval_checkpoint_-2') # Note: Check if it's eval_checkpoints_-2 or eval_checkpoint_-2
    csv_path = os.path.join(performance_dir, 'performance.csv')
    
    if not os.path.exists(csv_path):
        print(f"Warning: Could not find CSV for {method} at {csv_path}")
        continue
        
    df = pd.read_csv(csv_path)
    
    for index, row in df.iterrows():
        episode_id = row['episode_id']
        
        # Look up garment_id from info.json
        info_json_path = os.path.join(performance_dir, f'episode_{episode_id}', 'step_0', 'info.json')
        try:
            with open(info_json_path, 'r') as f:
                info_data = json.load(f)
                garment_id = info_data.get('garment_id')
                category = id_to_category.get(garment_id)
        except Exception as e:
            print(f"Skipping episode {episode_id} for {method}: Could not read info.json - {e}")
            continue
            
        if not category:
            continue # Skip if garment_id isn't in our target list

        # ---> NEW: Append the episode ID to the specific method and garment list <---
        trial_counts[method][category].append(episode_id)

        # Parse stringified lists
        nc_list = ast.literal_eval(row['evaluation/normalised_coverage'])
        ni_list = ast.literal_eval(row['evaluation/normalised_improvement'])
        iou_list = ast.literal_eval(row['evaluation/max_IoU_to_flattened'])
        
        for step in steps_to_evaluate:
            # Safely get the value at the specific step (or the last available if episode ended early)
            idx = min(step, len(nc_list) - 1)
            
            # For metrics, it has to be the maximum value from the beginning to the examined steps
            nc_val = max(nc_list[:idx+1]) * 100
            ni_val = max(ni_list[:idx+1]) * 100
            iou_val = max(iou_list[:idx+1]) * 100
            
            # Check success criteria (NC and IoU must meet thresholds AT THE SAME TIME)
            is_90_80 = 1 if any(nc_list[i] >= 0.90 and iou_list[i] >= 0.80 for i in range(idx + 1)) else 0
            is_90_0  = 1 if any(nc_list[i] >= 0.90 for i in range(idx + 1)) else 0
            
            # Store for specific garment
            for cat in [category, 'Real Total']:
                results[method][step][cat]['NC'].append(nc_val)
                results[method][step][cat]['NI'].append(ni_val)
                results[method][step][cat]['MaxIoU'].append(iou_val)
                results[method][step][cat]['success_90_80'] += is_90_80
                results[method][step][cat]['success_90_0'] += is_90_0
                results[method][step][cat]['total'] += 1

# --- Step 1.5: Print the Trial Counts and Episode IDs ---
print("="*60)
print("TRIAL COUNTS & EPISODE IDs PER METHOD & GARMENT")
print("="*60)
for method, categories in trial_counts.items():
    print(f"\n{method}:")
    total_method_trials = 0
    for category, ep_ids in categories.items():
        count = len(ep_ids)
        # Display the count and the sorted list of episode IDs
        print(f"  - {category}: {count} episodes (IDs: {sorted(ep_ids)})")
        total_method_trials += count
    print(f"  > Total for {method}: {total_method_trials} episodes")
print("\n" + "="*60 + "\n")



In [ ]:
# --- Step 2: Formatting Function ---
def fmt(method, step, category):
    """Calculates Mean and Std, formats as 'Mean \pm Std' and formats SR."""
    data = results[method][step][category]
    if data['total'] == 0:
        return "N/A & N/A & N/A & 0/0 & 0/0"
    
    nc_m, nc_s = np.mean(data['NC']), np.std(data['NC'])
    ni_m, ni_s = np.mean(data['NI']), np.std(data['NI'])
    iou_m, iou_s = np.mean(data['MaxIoU']), np.std(data['MaxIoU'])
    sr1, sr2 = data['success_90_80'], data['success_90_0']
    tot = data['total']
    
    return f"{nc_m:.1f} \$\pm$ {nc_s:.1f} & {ni_m:.1f} \$\pm$ {ni_s:.1f} & {iou_m:.1f} \$\pm$ {iou_s:.1f} & {sr1}/{tot} & {sr2}/{tot}"

# --- Step 3: Generate the LaTeX Table ---
latex_str = r"""\begin{table*}[t!]
\caption{Real-world garment flattening performance of different pick-and-place (PnP) controllers... (Insert full caption here)}
\label{tab:real-worldflattening}
\centering
\resizebox{\textwidth}{!}{%
\renewcommand{\arraystretch}{1.2} 
\setlength{\tabcolsep}{2pt}
\begin{tabular}{c|ccccc|ccccc||ccccc}
Method & \multicolumn{5}{c|}{PlaNet-ClothPick \cite{kadi2024planet}}  & \multicolumn{5}{c||}{LaGarNet} & \multicolumn{5}{c}{Human Policy} \\
\cline{1-16}
10 Steps & NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$    & SR$^{90}_{80}$  $\uparrow$  & SR$^{90}_{0}$  $\uparrow$ 
& NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$     & SR$^{90}_{80}$  $\uparrow$  & SR$^{90}_{0}$  $\uparrow$ 
& NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$    & SR$^{90}_{80}$  $\uparrow$  & SR$^{90}_{0}$  $\uparrow$ \\
\hline
"""

# Generate 10 Steps Rows
for cat in ['Longsleeve', 'Trousers', 'Dress', 'Skirt']:
    row = f"{cat} & " + " & ".join([fmt(m, 10, cat) for m in ['PlaNet-ClothPick', 'LaGarNet', 'Human']]) + r" \\" + "\n"
    latex_str += row

latex_str += r"\hline" + "\n"
latex_str += f"Real Total & " + " & ".join([fmt(m, 10, 'Real Total') for m in ['PlaNet-ClothPick', 'LaGarNet', 'Human']]) + r" \\" + "\n"

latex_str += r"""\hline
\hline
20 Steps 
& NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$    & SR$^{90}_{80}$  $\uparrow$  & SR$^{90}_{0}$  $\uparrow$ 
& NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$     & SR$^{90}_{80}$  $\uparrow$  & SR$^{90}_{0}$  $\uparrow$ 
& NC $\uparrow$  & NI $\uparrow$ & Max IoU $\uparrow$    & SR$^{90}_{80}$  $\uparrow$  & SR$^{90}_{0}$  $\uparrow$ \\
\hline
"""

# Generate 20 Steps Rows
for cat in ['Longsleeve', 'Trousers', 'Dress', 'Skirt']:
    row = f"{cat} & " + " & ".join([fmt(m, 20, cat) for m in ['PlaNet-ClothPick', 'LaGarNet', 'Human']]) + r" \\" + "\n"
    latex_str += row

latex_str += r"\hline" + "\n"
latex_str += f"Real Total & " + " & ".join([fmt(m, 20, 'Real Total') for m in ['PlaNet-ClothPick', 'LaGarNet', 'Human']]) + r" \\" + "\n"

latex_str += r"""\bottomrule
\end{tabular}
}
\end{table*}"""

print(latex_str)